___
# <center>Regressão linear múltipla e regressão logística</center>
___

## Aula 20

**Objetivo da aula:** ao final desta aula, você deve ser capaz de:

 * ler a saída de uma regressão no `statsmodels`: coeficiente, intervalo,
   valor-p e $R^2$;
 * interpretar cada coeficiente de uma regressão múltipla **mantidas as demais
   variáveis**, e explicar por que ele muda quando outra variável entra;
 * interpretar variáveis de sim ou não e de categorias (dummies);
 * ajustar uma regressão logística e ler $e^{b}$ como **razão de chances**;
 * calcular a probabilidade prevista para um perfil de processo.

**Primeira hora:** regressão linear múltipla, retomando a aula 19 com calma.
**Segunda hora:** regressão logística, na mesma base.

___
<div id="indice"></div>

## Índice

- [A base](#base)
- **Primeira hora: regressão linear**
  - [1. Regressão simples](#simples)
  - [2. Valor-p e R² respondem perguntas diferentes](#r2)
  - [3. Regressão múltipla](#multipla)
  - [4. Variáveis de sim ou não](#binaria)
  - [5. Variáveis com categorias](#categorias)
  - [6. Exercício: a duração do processo](#ex-linear)
- **Segunda hora: regressão logística**
  - [7. Desfecho de sim ou não](#sim-nao)
  - [8. Chance e razão de chances](#chance)
  - [9. Regressão logística](#logistica)
  - [10. Probabilidade prevista](#previsao)
  - [11. Exercício: escritórios de massa fecham mais acordos?](#ex-logistica)

___
<div id="base"></div>

# A base

**Atenção: a base é simulada.** Ela imita a carteira de ações revisionais de
financiamento de veículo que vimos com o BTG, mas cada linha foi sorteada no
computador, com efeitos escolhidos para a aula. Por isso ela é pública e vem
direto da internet, sem baixar nada. Os resultados ilustram os métodos e **não
dizem nada sobre processos reais**.

Cada linha é um processo já encerrado, em que o banco pagou alguma coisa ao
autor (por acordo ou por condenação).

| coluna | o que é |
|---|---|
| `tribunal` | TJSP, TJMG, TJRS, TJPR ou TJBA |
| `juizado` | 1 se tramitou no juizado especial, 0 se na vara comum |
| `escritorio_massa` | 1 se o advogado do autor é de escritório com muitas ações iguais |
| `idade_autor` | idade do autor, em anos |
| `valor_financiado` | valor financiado no contrato, em R$ mil |
| `taxa_juros` | taxa de juros do contrato, em % ao mês |
| `valor_causa` | valor da causa (o que o autor pediu), em R$ mil |
| `pericia` | 1 se houve perícia contábil |
| `acordo` | 1 se terminou em acordo, 0 se em sentença |
| `valor_pago` | quanto o banco pagou, em R$ mil |
| `duracao_meses` | meses do ajuizamento ao fim do processo |

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

processos = pd.read_csv("https://raw.githubusercontent.com/jtrecenti/202662-cdad2/main/dados/processos_simulados.csv")
processos.head()

In [ ]:
processos.describe().round(1)

`describe()` resume cada coluna numérica. Nas colunas de 0 e 1, a média é a
proporção de 1: por exemplo, a média de `acordo` é a fração de processos que
terminaram em acordo.

[Volta ao Índice](#indice)

___
<div id="simples"></div>

# Primeira hora: regressão linear

## 1. Regressão simples

A pergunta da primeira hora: **o que está associado ao valor pago pelo banco?**
Começamos com uma variável só: o valor da causa.

In [ ]:
processos.plot.scatter(x="valor_causa", y="valor_pago", alpha=0.3)

In [ ]:
modelo1 = smf.ols("valor_pago ~ ____", data=processos).fit()
modelo1.summary()

Onde olhar na saída:

* **`coef` de `valor_causa`:** quanto o valor pago muda, em média, quando o
  valor da causa aumenta 1 unidade. As duas colunas estão em R$ mil, então 0,313
  quer dizer: **cada R$ 1.000 a mais na causa estão associados a cerca de R$ 313
  a mais pagos**.
* **`[0.025  0.975]`:** o intervalo de 95% do coeficiente, de R$ 298 a R$ 327.
* **`P>|t|`:** o valor-p. Perto de zero: se não houvesse associação nenhuma,
  seria muito difícil ver um coeficiente desse tamanho por acaso.
* **`Intercept`:** o valor pago previsto para uma causa de R$ 0. Fora dos
  dados, sem leitura prática.
* **`R-squared`, no alto à direita:** 0,637. A reta explica 64% da variação do
  valor pago entre os processos.

In [ ]:
# a previsão da reta para uma causa de R$ 40 mil
modelo1.predict(pd.DataFrame({"valor_causa": [40]}))

[Volta ao Índice](#indice)

___
<div id="r2"></div>

## 2. Valor-p e R² respondem perguntas diferentes

Três modelos simples, cada um com uma explicativa. Compare o coeficiente, o
valor-p e o $R^2$.

In [ ]:
modelo_idade = smf.ols("valor_pago ~ idade_autor", data=processos).fit()
modelo_idade.summary()

In [ ]:
modelo_juros = smf.ols("valor_pago ~ taxa_juros", data=processos).fit()
modelo_juros.summary()

| explicativa | o coeficiente é diferente de zero? | $R^2$ |
|---|---|---|
| valor da causa | sim (valor-p < 0,001) | 0,64: explica muito |
| idade do autor | não (valor-p = 0,41) | 0,001: não explica nada |
| taxa de juros | sim (valor-p < 0,001) | 0,05: explica pouco |

* O **valor-p** pergunta: *existe* associação, ou o que vemos pode ser acaso?
* O **$R^2$** pergunta: *quanto* da variação do valor pago a variável explica?

A taxa de juros está associada ao valor pago, mas explica pouco: entre
processos com a mesma taxa, os valores ainda variam muito. **R² baixo não quer dizer coeficiente
inútil.**

[Volta ao Índice](#indice)

___
<div id="multipla"></div>

## 3. Regressão múltipla

Antes de pôr várias variáveis no modelo, a correlação entre elas.

In [ ]:
colunas = ["valor_pago", "valor_causa", "valor_financiado", "taxa_juros", "idade_autor"]
processos[colunas].corr().round(2)

O valor da causa tem correlação de 0,76 com o valor financiado: contratos
maiores geram pedidos maiores. E o financiado também está associado ao valor
pago. Então parte do coeficiente da causa, no modelo simples, é "tamanho do
contrato".

In [ ]:
modelo2 = smf.ols("valor_pago ~ valor_causa + ____", data=processos).fit()
modelo2.summary()

O coeficiente da causa caiu de 0,313 para 0,267. Agora ele vale **mantido o
valor financiado**: entre processos com o mesmo valor financiado, cada R$ 1.000
a mais na causa estão associados a cerca de R$ 267 a mais pagos.

Agora com todas as variáveis numéricas:

In [ ]:
modelo3 = smf.ols("valor_pago ~ valor_causa + valor_financiado + taxa_juros + idade_autor",
                  data=processos).fit()
modelo3.summary()

**Como relatar cada coeficiente:** a unidade, o "mantidas as demais" e o
intervalo.

* `valor_causa`: "Mantidos o valor financiado, a taxa de juros e a idade do
  autor, cada R$ 1.000 a mais na causa estão associados a cerca de R$ 247 a mais
  pagos (IC 95%: R$ 223 a R$ 271)."
* `taxa_juros`: cada ponto percentual a mais de juros ao mês, mantidas as
  demais, está associado a cerca de R$ 1.340 a mais pagos.
* `idade_autor`: valor-p de 0,32 e intervalo que inclui zero. **Não há
  evidência** de associação com o valor pago.

**R² e R² ajustado.** O $R^2$ foi de 0,647 para 0,653. Ele nunca cai quando uma
variável entra, mesmo que ela não ajude; o ajustado desconta o número de
variáveis.

[Volta ao Índice](#indice)

___
<div id="binaria"></div>

## 4. Variáveis de sim ou não

`acordo` vale 0 ou 1. Na regressão, o coeficiente de uma variável assim é a
**diferença de médias** entre os dois grupos: é o teste t, escrito como
regressão.

In [ ]:
processos.groupby("acordo")["valor_pago"].mean().round(2)

In [ ]:
modelo_acordo = smf.ols("valor_pago ~ acordo", data=processos).fit()
modelo_acordo.summary()

O intercepto (11,74) é a média de quem **não** fez acordo, e o coeficiente
(-3,86) é a diferença: nos acordos o banco paga, em média, cerca de R$ 3.860 a
menos.

**Juizado especial.** Agora a mesma conta para o juizado, sem e com o valor da
causa no modelo.

In [ ]:
sem_causa = smf.ols("valor_pago ~ juizado", data=processos).fit()
com_causa = smf.ols("valor_pago ~ juizado + ____", data=processos).fit()
print(sem_causa.params.round(3))
print(com_causa.params.round(3))
print(com_causa.pvalues.round(3))

Sozinho, o juizado paga R$ 5.300 a menos. Com o valor da causa no modelo, a
diferença cai para R$ 300, com valor-p de 0,28: **some**. As causas do juizado
são menores (ele só aceita causas até 40 salários mínimos), e era isso que
aparecia na comparação simples.

In [ ]:
processos.groupby("juizado")[["valor_causa", "valor_pago"]].mean().round(1)

[Volta ao Índice](#indice)

___
<div id="categorias"></div>

## 5. Variáveis com categorias

O tribunal tem cinco categorias. Na regressão, ele vira **quatro colunas de 0 e
1** (dummies), e o tribunal que fica de fora é a **referência**. Escolhemos o
TJSP, o maior.

In [ ]:
processos.groupby("tribunal")["valor_pago"].agg(["size", "mean"]).round(2)

In [ ]:
modelo_trib = smf.ols('valor_pago ~ C(tribunal, Treatment("____"))', data=processos).fit()
modelo_trib.summary()

* O **intercepto** (9,36) é a média do TJSP.
* Cada coeficiente é a **diferença para o TJSP**: no TJRS o banco paga, em
  média, R$ 2.320 a mais (valor-p < 0,001); no TJBA, R$ 1.730 a menos (valor-p
  = 0,002).
* TJMG e TJPR têm valor-p acima de 0,3: não se distinguem do TJSP.

Comparar a média de várias categorias de uma vez é a ANOVA da aula 17, aqui
escrita como regressão.

[Volta ao Índice](#indice)

___
<div id="ex-linear"></div>

## 6. Exercício: a duração do processo

Agora a variável resposta é `duracao_meses`. Em duplas.

In [ ]:
duracao1 = smf.ols("duracao_meses ~ valor_causa", data=processos).fit()
duracao1.summary()

In [ ]:
duracao2 = smf.ols('duracao_meses ~ juizado + pericia + C(tribunal, Treatment("TJSP"))',
                   data=processos).fit()
duracao2.summary()

**a)** No primeiro modelo, o valor da causa está associado à duração? Quanto ele
explica?

**b)** No segundo, escreva a frase do coeficiente do juizado, com unidade e
"mantidas as demais".

**c)** Qual tribunal é o mais lento, mantidos juizado e perícia? Algum não se
distingue do TJSP?

**d)** Compare os dois $R^2$. Qual modelo explica melhor a duração?

*(escreva aqui)*

[Volta ao Índice](#indice)

___
<div id="sim-nao"></div>

# Segunda hora: regressão logística

## 7. Desfecho de sim ou não

A pergunta da segunda hora: **o que está associado a um processo terminar em
acordo?** `acordo` é 1 ou 0. A regressão linear roda, mas não sabe que a
resposta é uma probabilidade.

In [ ]:
reta = smf.ols("acordo ~ valor_causa", data=processos).fit()
reta.predict(pd.DataFrame({"valor_causa": [5, 30, 60, 100]})).round(3)

Para uma causa de R$ 100 mil, a reta prevê uma probabilidade de acordo
**negativa** (-4%). A regressão logística modela a chance numa escala em que a
curva fica sempre entre 0 e 1.

[Volta ao Índice](#indice)

___
<div id="chance"></div>

## 8. Chance e razão de chances

$$\text{chance} = \frac{p}{1 - p}$$

Probabilidade de 80% é chance de 4: 4 acordos para cada processo que não fez
acordo.

In [ ]:
pd.crosstab(processos["juizado"], processos["acordo"], margins=True)

In [ ]:
chance_juizado = 206 / 63            # acordos / sem acordo, no juizado
chance_vara = ____ / ____            # acordos / sem acordo, na vara comum
print(f"chance no juizado: {chance_juizado:.2f}")
print(f"chance na vara: {chance_vara:.2f}")
print(f"razão de chances: {chance_juizado / chance_vara:.2f}")

No juizado, a chance de acordo é **3,66 vezes** a da vara comum. Em
probabilidade: 77% contra 47%.

[Volta ao Índice](#indice)

___
<div id="logistica"></div>

## 9. Regressão logística

$$\log\left(\frac{p}{1-p}\right) = a + b_1 x_1 + b_2 x_2 + \dots$$

O coeficiente $b$ está na escala do log da chance. **$e^{b}$ é a razão de
chances**: quantas vezes a chance é multiplicada quando $x$ aumenta 1 unidade,
mantidas as demais.

Com o juizado sozinho, o modelo reproduz a conta da tabela:

In [ ]:
logit1 = smf.logit("acordo ~ juizado", data=processos).fit()
np.exp(logit1.____)

Agora com várias variáveis:

In [ ]:
logit2 = smf.logit("acordo ~ juizado + escritorio_massa + valor_causa + idade_autor",
                   data=processos).fit()
logit2.summary()

In [ ]:
# razões de chances e intervalos de 95%
print(np.exp(logit2.params).round(3))
print(np.exp(logit2.conf_int()).round(3))

**Como relatar:**

* `juizado`: mantidos o escritório, o valor da causa e a idade, a chance de
  acordo no juizado é **2,57 vezes** a da vara comum (IC 95%: 1,79 a 3,67). Era
  3,66 sozinho: parte da diferença vinha das causas menores do juizado.
* `escritorio_massa`: a chance de acordo é 1,92 vez a dos demais escritórios.
* `valor_causa`: cada R$ 1.000 a mais **multiplicam** a chance por 0,975, ou
  seja, 2,5% a menos. Para R$ 10 mil a mais: $0{,}975^{10} \approx 0{,}78$.
* `idade_autor`: razão de 1,001, valor-p de 0,87. Sem evidência de associação.

Razão de chances **acima de 1** aumenta a chance; **abaixo de 1** diminui;
**igual a 1** não muda nada.

[Volta ao Índice](#indice)

___
<div id="previsao"></div>

## 10. Probabilidade prevista

O modelo devolve a probabilidade de acordo para qualquer perfil de processo.

In [ ]:
perfis = pd.DataFrame({
    "juizado":          [1,  0,  0],
    "escritorio_massa": [1,  1,  0],
    "valor_causa":      [15, 15, 50],
    "idade_autor":      [40, 40, 40],
})
perfis["prob_acordo"] = logit2.____(perfis).round(3)
perfis

Uma causa pequena no juizado, com escritório de massa: 85%. A mesma causa na
vara comum: 69%. Uma causa de R$ 50 mil na vara, com outro escritório: 33%.

A probabilidade prevista é o que vira decisão: na aula 21 vamos escolher um
**ponto de corte** e contar os erros de cada lado.

[Volta ao Índice](#indice)

___
<div id="ex-logistica"></div>

## 11. Exercício: escritórios de massa fecham mais acordos?

Em duplas. Siga as etapas e escreva o relatório no fim.

**1. Exploratória.** A proporção de acordos com e sem escritório de massa, com o
número de processos.

In [ ]:
processos.groupby("____")["acordo"].agg(["size", "mean"]).round(3)

**2. Modelo.** Regressão logística com o escritório, ajustada pelo juizado, pelo
valor da causa e pelo tribunal (referência: TJSP). Calcule as razões de chances.

In [ ]:
logit3 = smf.logit('acordo ~ escritorio_massa + juizado + valor_causa + C(tribunal, Treatment("TJSP"))',
                   data=processos).fit()
np.exp(____).round(3)

In [ ]:
np.exp(logit3.conf_int()).round(3)

**3. Previsão.** A probabilidade de acordo de uma causa de R$ 30 mil na vara
comum do TJSP, com e sem escritório de massa.

In [ ]:
novos = pd.DataFrame({"escritorio_massa": [1, 0], "juizado": [0, 0],
                      "valor_causa": [30, 30], "tribunal": ["TJSP", "TJSP"]})
logit3.predict(novos).round(3)

**4. Relatório**, em três frases, na célula abaixo: a pergunta; o resultado,
com a razão de chances, o intervalo e "mantidos..."; e **uma** limitação (o que
o modelo não permite concluir).

*(escreva aqui)*

[Volta ao Índice](#indice)